# Test

In [1]:
# pip install torch transformers pillow

import torch
import torch.nn as nn
from PIL import Image
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoProcessor, AutoModel

DEVICE = "cuda"

# --------- LOAD MODELS (FROZEN) ---------

LLM_NAME = "Qwen/Qwen2.5-0.5B"
VISION_NAME = "google/siglip-base-patch16-224"

tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_NAME, torch_dtype=torch.float16
).to(DEVICE)
llm.requires_grad_(False)
llm.eval()

vision_processor = AutoProcessor.from_pretrained(VISION_NAME)
vision_model = AutoModel.from_pretrained(
    VISION_NAME, torch_dtype=torch.float16
).to(DEVICE)
vision_model.requires_grad_(False)
vision_model.eval()

# --------- ADD IMAGE TOKEN ---------

IMAGE_TOKEN = "<image>"
if IMAGE_TOKEN not in tokenizer.get_vocab():
    tokenizer.add_special_tokens({"additional_special_tokens": [IMAGE_TOKEN]})
    llm.resize_token_embeddings(len(tokenizer))

# --------- CONNECTOR (WIDE + SHALLOW) ---------

vision_dim = vision_model.config.hidden_size
llm_dim = llm.config.hidden_size

connector = nn.Sequential(
    nn.Linear(vision_dim, 4096),
    nn.GELU(),
    nn.Linear(4096, llm_dim)
).to(DEVICE)

optimizer = torch.optim.AdamW(connector.parameters(), lr=1e-3)

# --------- IMAGE → VISUAL TOKENS ---------

def encode_image(img, K=32):
    inputs = vision_processor(images=img, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        feats = vision_model(**inputs).last_hidden_state
    feats = feats[:, :K, :]              # simple truncation
    return connector(feats)              # (1, K, D_llm)

# --------- BUILD MULTIMODAL EMBEDDINGS ---------

def build_inputs(img, text):
    vis = encode_image(img)
    text = f"{IMAGE_TOKEN} {text}"
    ids = tokenizer(text, return_tensors="pt").input_ids.to(DEVICE)
    embeds = llm.get_input_embeddings()(ids)

    idx = (ids == tokenizer.convert_tokens_to_ids(IMAGE_TOKEN)).nonzero()[0,1]
    embeds = torch.cat([embeds[:,:idx], vis, embeds[:,idx+1:]], dim=1)
    return embeds

# --------- ONE TRAIN STEP (ALIGNMENT) ---------

img = Image.open("example.jpg").convert("RGB")
caption = "A dog running in a grassy field."

labels = tokenizer(caption, return_tensors="pt").input_ids.to(DEVICE)
inputs_embeds = build_inputs(img, caption)

out = llm(inputs_embeds=inputs_embeds, labels=labels)
loss = out.loss
loss.backward()

optimizer.step()
optimizer.zero_grad()

print("loss:", loss.item())

# --------- INFERENCE TEST ---------

with torch.no_grad():
    prompt = "What is in this image?"
    embeds = build_inputs(img, prompt)
    out = llm.generate(inputs_embeds=embeds, max_new_tokens=40)

print(tokenizer.decode(out[0], skip_special_tokens=True))


ModuleNotFoundError: No module named 'transformers'